In [0]:
# Load the EV charging station data
# Replace the table name below with the correct one if needed
ladestation_df = spark.table("`germany's_ev_charging_infrastructure_status`.default.ladestation_fact_table_1")

# Group by commissioning date and count stations
trend_df = (
    ladestation_df
    .groupBy("Inbetriebnahmedatum")
    .count()
    .orderBy("Inbetriebnahmedatum")
    .withColumnRenamed("count", "number_of_stations")
)

# Display the trend
display(trend_df)

In [0]:
import matplotlib.pyplot as plt

trend_pd = trend_df.toPandas()

plt.figure(figsize=(12, 6))
plt.plot(trend_pd["Inbetriebnahmedatum"], trend_pd["number_of_stations"], marker='o')
plt.xlabel("Inbetriebnahmedatum")
plt.ylabel("Number of Stations")
plt.title("Trend of EV Charging Stations Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
import plotly.express as px

trend_pd = trend_df.toPandas()

fig = px.line(
    trend_pd,
    x="Inbetriebnahmedatum",
    y="number_of_stations",
    markers=True,
    title="Trend of EV Charging Stations Over Time",
    labels={"Inbetriebnahmedatum": "Inbetriebnahmedatum", "number_of_stations": "Number of Stations"}
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

The trend visualizations show how the number of EV charging stations commissioned in Germany has changed over time.
An upward trend indicates increasing deployment of charging infrastructure, supporting EV adoption.
Peaks or dips may correspond to policy changes, funding cycles, or market dynamics.
Consistent growth suggests ongoing investment, while plateaus or declines may signal market saturation or regulatory barriers.
For deeper insights, correlate these trends with external factors such as government incentives, EV sales, or regional policies.

In [0]:
from pyspark.sql.functions import when, col

energy_grouped_df = ladestation_df.withColumn(
    "energy_output_group",
    when(col("NennleistungBNetzA") < 22, "<22kW -Standard AC chargers")
    .when((col("NennleistungBNetzA") >= 22) & (col("NennleistungBNetzA") <= 50), "23-50kW - Faster DC chargers")
    .when((col("NennleistungBNetzA") > 50) & (col("NennleistungBNetzA") <= 150), "51-150kW - Faster DC chargers")
    .when(col("NennleistungBNetzA") > 150, "HPC (High Power Chargers, >150kW")
)

grouped_count_df = (
    energy_grouped_df
    .groupBy("energy_output_group")
    .count()
    .orderBy("energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

display(grouped_count_df)

In [0]:
energy_trend_df = (
    energy_grouped_df
    .groupBy("Inbetriebnahmedatum", "energy_output_group")
    .count()
    .orderBy("Inbetriebnahmedatum", "energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

display(energy_trend_df)

In [0]:
import plotly.express as px
from pyspark.sql.functions import year

energy_trend_year_df = (
    energy_grouped_df
    .withColumn("year", year(col("Inbetriebnahmedatum")))
    .groupBy("year", "energy_output_group")
    .count()
    .orderBy("year", "energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

energy_trend_year_pd = energy_trend_year_df.toPandas()

fig = px.line(
    energy_trend_year_pd,
    x="year",
    y="number_of_stations",
    color="energy_output_group",
    markers=True,
    title="Growth of Energy Output Charger Over Time (Yearly)",
    labels={"year": "Year", "number_of_stations": "Number of Stations", "energy_output_group": "Energy Output Group"}
)
fig.update_layout(xaxis_tickangle=0)
fig.show()

Interpretation & Context:

  The chart shows a clear increase in all charger types, with rapid growth in HPCs in recent years.
  The expansion of HPCs is particularly significant for Germany’s sustainability goals, as it supports long-range EV travel and reduces charging time, making EVs more practical for everyone.